In [1]:
from google.colab import files
files.upload('kaggle')

Saving kaggle.json to kaggle/kaggle.json


{'kaggle/kaggle.json': b'{\r\n  "username": "Rahmandika",\r\n  "key": "KGAT_ed3162db845696b91a5e8c09f656f8ff"\r\n}\r\n'}

In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [3]:
!kaggle datasets download -d danielbacioiu/tig-aluminium-5083

Dataset URL: https://www.kaggle.com/datasets/danielbacioiu/tig-aluminium-5083
License(s): CC-BY-SA-4.0
100% 11.2G/11.2G [01:45<00:00, 114MB/s]



In [4]:
!unzip -q tig-aluminium-5083.zip -d /content/welding_dataset

In [5]:
import os
import json
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import matplotlib.pyplot as plt

# Verifikasi GPU (jika menggunakan Google Colab agar training lebih cepat)
print("TensorFlow Version:", tf.__version__)
print("GPU Tersedia:" if tf.config.list_physical_devices('GPU') else "GPU Tidak Tersedia, menggunakan CPU.")

TensorFlow Version: 2.20.0
GPU Tidak Tersedia, menggunakan CPU.


In [6]:
import os
import json
import pandas as pd

# Path dasar menuju tempat folder train dan test berada
BASE_DATASET_PATH = "welding_dataset/al5083"

def json_to_dataframe(json_path, folder_gambar_path):
    if not os.path.exists(json_path):
        print(f"Error: File JSON tidak ditemukan di -> {json_path}")
        return pd.DataFrame()

    with open(json_path, 'r') as file:
        data = json.load(file)

    list_paths = []
    list_labels = []

    for relative_path, label_code in data.items():
        # Gabungkan path folder dengan nama file gambar dari JSON
        full_path = os.path.join(folder_gambar_path, relative_path)

        # Validasi fisik apakah gambarnya ada di dalam folder train/test
        if os.path.exists(full_path):
            list_paths.append(full_path)
            list_labels.append(str(label_code))

    df = pd.DataFrame({
        'image_path': list_paths,
        'label': list_labels
    })
    return df

# Konfigurasi jalur file untuk Data Training
path_train_json = os.path.join(BASE_DATASET_PATH, "train", "train.json")
folder_gambar_train = os.path.join(BASE_DATASET_PATH, "train")

# Konfigurasi jalur file untuk Data Testing
path_test_json = os.path.join(BASE_DATASET_PATH, "test", "test.json")
folder_gambar_test = os.path.join(BASE_DATASET_PATH, "test")

# Eksekusi pembuatan DataFrame
df_train = json_to_dataframe(path_train_json, folder_gambar_train)
df_test = json_to_dataframe(path_test_json, folder_gambar_test)

print("=== VERIFIKASI DATASET ===")
print(f"Jumlah gambar TRAINING yang siap digunakan : {len(df_train)} gambar")
print(f"Jumlah gambar TESTING yang siap digunakan  : {len(df_test)} gambar")

if not df_train.empty:
    print("\n=== Sampel Data Tabel Training ===")
    print(df_train.head())

=== VERIFIKASI DATASET ===
Jumlah gambar TRAINING yang siap digunakan : 26666 gambar
Jumlah gambar TESTING yang siap digunakan  : 6588 gambar

=== Sampel Data Tabel Training ===
                                          image_path label
0  welding_dataset/al5083/train/170906-113317-Al ...     1
1  welding_dataset/al5083/train/170906-113317-Al ...     1
2  welding_dataset/al5083/train/170906-113317-Al ...     1
3  welding_dataset/al5083/train/170906-113317-Al ...     1
4  welding_dataset/al5083/train/170906-113317-Al ...     1


In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_HEIGHT = 150
IMG_WIDTH = 150
BATCH_SIZE = 32

# Generator dengan normalisasi nilai piksel (rescale 1./255)
train_datagen = ImageDataGenerator(rescale=1./255, horizontal_flip=True, fill_mode='nearest')
test_datagen = ImageDataGenerator(rescale=1./255)

# Menghubungkan generator dengan DataFrame Training
train_generator = train_datagen.flow_from_dataframe(
    dataframe=df_train,
    x_col='image_path',
    y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

# Menghubungkan generator dengan DataFrame Testing
test_generator = test_datagen.flow_from_dataframe(
    dataframe=df_test,
    x_col='image_path',
    y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 26666 validated image filenames belonging to 6 classes.
Found 6588 validated image filenames belonging to 6 classes.
